# Episode 14: Workflow & TransitionGraph

`discussion` cycles in a fixed order. Real orchestration needs *conditions* — "after the drafter, go to the reviewer", "if approved, stop". That's the `workflow` adapter.

**In this episode you'll build:** a draft → review → polish pipeline driven by a declarative graph.

## What `workflow` does

`workflow` is the **orchestrated** N-party adapter. Turn-taking is described by a declarative **`TransitionGraph`**:

- `initial_speaker` — who goes first
- `transitions` — a list of `Transition(when=<condition>, then=<target>)` rules
- `default_target` — used if no transition matches
- `max_turns` — a hard cap

On every turn the adapter walks the transitions; the first matching condition's target picks the next speaker. It's the modern replacement for the classic `GroupChat` + handoffs.

## Targets and conditions

**Targets** (the `then=`) decide who speaks next:

| Target | Meaning |
|---|---|
| `AgentTarget(id)` | Hand to a specific agent |
| `RoundRobinTarget()` | Advance through the participant order |
| `RevertToInitiatorTarget()` | Hand back to whoever opened the channel |
| `TerminateTarget(reason)` | End the channel |

**Conditions** (the `when=`) decide *if* a transition fires:

| Condition | Fires when |
|---|---|
| `Always()` | Every turn |
| `FromSpeaker(id)` | The last message came from this agent |
| `ToolCalled(name)` | The last turn called this tool |
| `ContextEquals(key, value)` | A channel context variable matches |

## Setup

`TransitionGraph.sequence([...])` is a factory that builds a simple linear pipeline for you. Workflow agent turns are recorded as `EV_PACKET` envelopes (not `EV_TEXT`), so our transcript helper handles both.

In [ ]:
from dotenv import load_dotenv

load_dotenv()

import asyncio

from autogen.beta import Agent
from autogen.beta.config import OpenAIConfig
from autogen.beta.knowledge import MemoryKnowledgeStore
from autogen.beta.network import (
    EV_CHANNEL_CLOSED,
    EV_PACKET,
    EV_TEXT,
    Hub,
    HubClient,
    LocalLink,
    Passport,
    Resume,
    TransitionGraph,
)

config = OpenAIConfig(model="gpt-4.1-mini", temperature=0)


async def wait_for_close(hub, channel_id, timeout=240.0):
    deadline = asyncio.get_event_loop().time() + timeout
    while asyncio.get_event_loop().time() < deadline:
        for e in await hub.read_wal(channel_id):
            if e.event_type == EV_CHANNEL_CLOSED:
                return e
        await asyncio.sleep(0.1)
    raise asyncio.TimeoutError("no close")


def turn_text(env):
    """Workflow turns are EV_PACKET; the kickoff is EV_TEXT."""
    if env.event_type == EV_TEXT:
        return env.event_data.get("text")
    if env.event_type == EV_PACKET:
        return env.event_data.get("body")
    return None

## A draft → review → polish pipeline

One subtlety: the agent that *opens* the channel and sends the kickoff has that send counted as its turn — its `Agent` never runs for it. So the kickoff is sent by a **`HumanClient`** (a non-LLM participant), and the first real agent — the drafter — comes after it in the graph.

In [ ]:
hub = await Hub.open(MemoryKnowledgeStore(), ttl_sweep_interval=0)
link = LocalLink(hub)
u_hc, d_hc, r_hc, p_hc = (HubClient(link, hub=hub) for _ in range(4))

# A HumanClient seeds the pipeline: its kickoff send IS its turn.
user = await u_hc.register_human(Passport(name="user", kind="human"))
drafter = await d_hc.register(
    Agent(
        "drafter", prompt="Write a 2-sentence first draft on the brief.", config=config
    ),
    Passport(name="drafter"),
    Resume(),
    attach_plugin=False,
)
reviewer = await r_hc.register(
    Agent(
        "reviewer",
        prompt="Suggest one concrete improvement to the draft.",
        config=config,
    ),
    Passport(name="reviewer"),
    Resume(),
    attach_plugin=False,
)
polisher = await p_hc.register(
    Agent(
        "polisher",
        prompt="Apply the suggestion and output the final 2 sentences.",
        config=config,
    ),
    Passport(name="polisher"),
    Resume(),
    attach_plugin=False,
)

# A declarative graph: user -> drafter -> reviewer -> polisher -> terminate.
graph = TransitionGraph.sequence(
    [user.agent_id, drafter.agent_id, reviewer.agent_id, polisher.agent_id]
)
channel = await user.open(
    type="workflow",
    target=[drafter.agent_id, reviewer.agent_id, polisher.agent_id],
    knobs={"graph": graph.to_dict()},
)
await channel.send("Brief: explain why code review matters, for a junior developer.")

close_env = await wait_for_close(hub, channel.channel_id)
print(f"closed: {close_env.event_data.get('reason')!r}")

names = {
    user.agent_id: "user",
    drafter.agent_id: "drafter",
    reviewer.agent_id: "reviewer",
    polisher.agent_id: "polisher",
}
for env in await hub.read_wal(channel.channel_id):
    text = turn_text(env)
    if text:
        print(f"[{names[env.sender_id]}] {text}")

for hc in (u_hc, d_hc, r_hc, p_hc):
    await hc.close()
await hub.close()

## What `sequence` built for you

`TransitionGraph.sequence([a, b, c, d])` is shorthand for a chain of `FromSpeaker(x) → AgentTarget(next)` transitions plus a `TerminateTarget("sequence_complete")` default. After the last agent speaks, no `FromSpeaker` rule matches, the default fires, and the channel closes — exactly the `sequence_complete` you saw.

For anything beyond a straight line, you write the `transitions` list yourself.

## Additional content

**The kickoff gotcha.** If the agent that opens the channel is also supposed to *produce* the first artifact, your pipeline breaks — the second agent receives the bare prompt as if it were the first agent's output. The fix used here: a `HumanClient` seeds the channel, and real agents come after it.

**Context variables.** A workflow channel has channel-scoped `context_vars`. A tool can write one, and a `ContextEquals(key, value)` condition can route on it — the basis for feedback loops (drafter ↔ reviewer until approved).

**`round_robin`.** `TransitionGraph.round_robin(participants=[...], max_turns=N)` reproduces Episode 13's discussion shape inside a workflow — handy when you'll later add conditions.

## What's next

A `sequence` graph is fixed. Episode 15 puts an *agent* in charge of routing — the **coordinator pattern**, the Beta replacement for the classic AutoPattern.